<a href="https://colab.research.google.com/github/Mukundan-T/seqADAGE/blob/master/Py/muk_transfer_learning/genomic_mapping/Mukundan_sA_ec_pg_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E. coli &rightarrow; S. aureus Gene Mapping

### Mukundan Thanigaivelan

#### July 9, 2026

Here we are trying to map E. coli genes to S. aureus genes using techniques such as BLAST, k-mer mapping, and AlphaFold.

## 1. Connect to GitHub

In [1]:
!git clone https://github.com/Mukundan-T/seqADAGE.git

fatal: destination path 'seqADAGE' already exists and is not an empty directory.


In [2]:
%cd seqADAGE/Py/muk_transfer_learning/genomic_mapping

/content/seqADAGE/Py/muk_transfer_learning/genomic_mapping


In [3]:
!pip install -qq biopython

## 2. Loading classes & modules

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [11]:
# Data Analysis
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Miscellaneous
import time
import tensorflow as tf

In [12]:
# check CPU and GPU available in runtime
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU')))
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print(tf.test.is_built_with_cuda())

Num CPUs Available:  1
Num GPUs Available:  1
True


## 3. BLAST

### Installation

In [15]:
!apt-get -qq update
!apt-get -qq install ncbi-blast+

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ncbi-data.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../ncbi-data_6.1.20170106+dfsg1-9_all.deb ...
Unpacking ncbi-data (6.1.20170106+dfsg1-9) ...
Selecting previously unselected package ncbi-blast+.
Preparing to unpack .../ncbi-blast+_2.12.0+ds-3build1_amd64.deb ...
Unpacking ncbi-blast+ (2.12.0+ds-3build1) ...
Setting up ncbi-data (6.1.20170106+dfsg1-9) ...
Setting up ncbi-blast+ (2.12.0+ds-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...


In [16]:
!blastn -version

blastn: 2.12.0+
 Package: blast 2.12.0, build Mar  8 2022 16:19:08


### Make BLAST database

In [23]:
ec_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa'
sa_fasta = '/content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/sa_pan_genome_reference.fa'

In [24]:
!makeblastdb -in "{ec_fasta}" -dbtype nucl -out ecoli_db



Building a new DB, current time: 07/10/2026 14:49:17
New DB name:   /content/seqADAGE/Py/muk_transfer_learning/genomic_mapping/ecoli_db
New DB title:  /content/drive/MyDrive/Comp-Bio-Projs-S26/data/muk-in-use/ec_pan_genome_reference.fa
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 68546 sequences in 3.00472 seconds.




### Blast S. aureus genes against database and save hits

In [37]:
blast_hits_file = '/content/drive/My Drive/Comp-Bio-Projs-S26/data/muk-in-use/blast_hits.tsv'

In [38]:
!blastn \
  -query "{sa_fasta}" \
  -db ecoli_db \
  -out "{blast_hits_file}" \
  -outfmt 6

In [40]:
columns = [
  "query", "subject", "pident", "length",
  "mismatch", "gapopen", "qstart", "qend",
  "sstart", "send", "evalue", "bitscore"
]

hits = pd.read_csv(blast_hits_file, sep = "\t", names = columns)
hits.shape

(7312, 12)

### Inspect and filter for best hits